## 1. Project Objective

This project investigates customer purchasing behavior in online retail
with the aim of understanding customer loyalty and identifying customers
who may become inactive.

This notebook focuses on:

- Loading the raw transaction dataset
- Assessing data quality
- Handling missing values
- Removing duplicate records
- Correcting data types
- Identifying cancelled transactions
- Handling invalid transactions
- Detecting potential outliers
- Creating meaningful features
- Generating descriptive statistics
- Exporting a cleaned dataset for further analysis

In [1]:
import pandas as pd
import numpy as np

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\Admin\Desktop\online-retail-customer-story\.venv\Scripts\python.exe

Python version:
3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]


## Load Dataset

In [3]:
file_path = "../data/raw/online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)

print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['Year 2009-2010', 'Year 2010-2011']


In [4]:
df_1 = pd.read_excel(file_path,sheet_name=0)
print("First sheet shape:", df_1.shape)


df_2 = pd.read_excel(file_path,sheet_name=1)
print("Second sheet shape:", df_2.shape)

#Combine
df = pd.concat([df_1, df_2],ignore_index=True
)
print("Combined dataset shape:", df.shape)

First sheet shape: (525461, 8)
Second sheet shape: (541910, 8)
Combined dataset shape: (1067371, 8)


## Initial Data Inspection

Before performing any cleaning, the dataset is inspected to understand
its structure, dimensions, variables, data types and overall quality.

In [5]:
print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)
print(df.shape)

print("\n" + "=" * 60)
print("COLUMN INFORMATION")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("DUPLICATES")
print("=" * 60)
print(df.duplicated().sum())

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(df.describe(include="all").T)

DATASET SHAPE
(1067371, 8)

COLUMN INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 65.1+ MB

MISSING VALUES
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

DUPLICATES
34335

SUMMARY STATISTICS
         

## Data Cleaning

The raw dataset is preserved in `df`. A separate working copy is created
for preprocessing so that the original imported data remains unchanged.

In [6]:
# Create a working copy of the raw dataset
df_clean = df.copy()

print("Raw dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

Raw dataset shape: (1067371, 8)
Working dataset shape: (1067371, 8)


### Standardizing Column Names

Column names are standardized to improve readability and make them easier
to reference during Python-based analysis.

In [7]:
df_clean = df_clean.rename(columns={
    "Invoice": "InvoiceNo",
    "Price": "UnitPrice",
    "Customer ID": "CustomerID"
})

print("Updated column names:")
print(df_clean.columns.tolist())

Updated column names:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


### Data Type Validation

In [8]:
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    errors="coerce"
)

df_clean["Quantity"] = pd.to_numeric(
    df_clean["Quantity"],
    errors="coerce"
)

df_clean["UnitPrice"] = pd.to_numeric(
    df_clean["UnitPrice"],
    errors="coerce"
)

df_clean["CustomerID"] = pd.to_numeric(
    df_clean["CustomerID"],
    errors="coerce"
)

print(df_clean.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


### Duplicate Removal

Exact duplicate transaction records are removed to prevent repeated
records from artificially inflating transaction counts and revenue.

In [9]:
duplicates_before = df_clean.duplicated().sum()

print("Duplicate records before removal:", duplicates_before)

df_clean = df_clean.drop_duplicates().copy()

duplicates_after = df_clean.duplicated().sum()

print("Duplicate records after removal:", duplicates_after)
print("Dataset shape after duplicate removal:", df_clean.shape)

rows_before_duplicates = len(df)

rows_after_duplicates = len(df_clean)

duplicates_removed = (
    rows_before_duplicates - rows_after_duplicates
)

print("Rows removed:", duplicates_removed)
print("Remaining rows:", rows_after_duplicates)

Duplicate records before removal: 34335
Duplicate records after removal: 0
Dataset shape after duplicate removal: (1033036, 8)
Rows removed: 34335
Remaining rows: 1033036


### Handling Missing Customer IDs

CustomerID is essential for customer-level analysis. Because CustomerID
is an identifier rather than a continuous measurement, missing values are
not imputed. Transactions without a CustomerID are excluded from the
customer-level analysis to avoid creating artificial customer identities.

In [10]:
#Analyze missing values
missing_summary = (
    df_clean.isnull()
    .sum()
    .to_frame("Missing_Count")
)

missing_summary["Missing_Percentage"] = (
    missing_summary["Missing_Count"]
    / len(df_clean) * 100
).round(2)

print(missing_summary)

             Missing_Count  Missing_Percentage
InvoiceNo                0                0.00
StockCode                0                0.00
Description           4275                0.41
Quantity                 0                0.00
InvoiceDate              0                0.00
UnitPrice                0                0.00
CustomerID          235151               22.76
Country                  0                0.00


In [11]:
#Treat CustomerID
customer_df = df_clean.dropna(
    subset=["CustomerID"]
).copy()

print("Rows before:", len(df_clean))
print("Rows after:", len(customer_df))

Rows before: 1033036
Rows after: 797885


In [12]:
#Missing Description treatment
customer_df["Description"] = (
    customer_df["Description"]
    .fillna("Unknown Product")
)

In [13]:
print("=" * 60)
print("CLEANING PROGRESS")
print("=" * 60)

print("Original rows:", len(df))
print("After duplicate removal:", len(df_clean))
print("Customer-linked rows:", len(customer_df))
print("Unique customers:", customer_df["CustomerID"].nunique())
print("Missing descriptions:", customer_df["Description"].isnull().sum())
print("Duplicates remaining:", customer_df.duplicated().sum())

CLEANING PROGRESS
Original rows: 1067371
After duplicate removal: 1033036
Customer-linked rows: 797885
Unique customers: 5942
Missing descriptions: 0
Duplicates remaining: 0


### Cancellation Analysis

Online retail transactions may contain cancelled or returned items.
According to the dataset documentation, invoice numbers beginning with
"C" indicate cancellations.


In [14]:
# Remove cancelled transactions
cancelled = customer_df["InvoiceNo"].astype(str).str.startswith("C")

print("Cancelled transactions:", cancelled.sum())

customer_df = customer_df[~cancelled].copy()

print("Rows after removing cancellations:", len(customer_df))

Cancelled transactions: 18390
Rows after removing cancellations: 779495


In [15]:
# Remove invalid sales transactions
customer_df = customer_df[
    (customer_df["Quantity"] > 0) &
    (customer_df["UnitPrice"] > 0)
].copy()

print("Rows after removing invalid transactions:", len(customer_df))

Rows after removing invalid transactions: 779425


## Outlier Detection
The IQR method is used to identify potential outliers in Quantity and UnitPrice. Potential outliers are flagged rather than automatically removed because some may represent genuine bulk purchases.

In [17]:
Q1 = customer_df[["Quantity", "UnitPrice"]].quantile(0.25)
Q3 = customer_df[["Quantity", "UnitPrice"]].quantile(0.75)

IQR = Q3 - Q1

outlier_mask = (
    (customer_df["Quantity"] < Q1["Quantity"] - 1.5 * IQR["Quantity"]) |
    (customer_df["Quantity"] > Q3["Quantity"] + 1.5 * IQR["Quantity"]) |
    (customer_df["UnitPrice"] < Q1["UnitPrice"] - 1.5 * IQR["UnitPrice"]) |
    (customer_df["UnitPrice"] > Q3["UnitPrice"] + 1.5 * IQR["UnitPrice"])
)

print("Potential outliers:", outlier_mask.sum())

Potential outliers: 116052


## Feature Engineering

Two new features are created to support customer-level analysis:
Revenue and PurchaseMonth.

In [18]:
# Feature 1: Revenue
customer_df["Revenue"] = (
    customer_df["Quantity"] * customer_df["UnitPrice"]
)

# Feature 2: Purchase Month
customer_df["PurchaseMonth"] = (
    customer_df["InvoiceDate"].dt.to_period("M").astype(str)
)

print(customer_df[
    ["Quantity", "UnitPrice", "Revenue", "PurchaseMonth"]
].head())

   Quantity  UnitPrice  Revenue PurchaseMonth
0        12       6.95    83.40       2009-12
1        12       6.75    81.00       2009-12
2        12       6.75    81.00       2009-12
3        48       2.10   100.80       2009-12
4        24       1.25    30.00       2009-12


## Descriptive Statistics

In [19]:
print("=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)

print(
    customer_df[
        ["Quantity", "UnitPrice", "Revenue"]
    ].describe()
)

DESCRIPTIVE STATISTICS
        Quantity  UnitPrice    Revenue
count 779,425.00 779,425.00 779,425.00
mean       13.49       3.22      22.29
std       145.86      29.68     227.43
min         1.00       0.00       0.00
25%         2.00       1.25       4.95
50%         6.00       1.95      12.48
75%        12.00       3.75      19.80
max    80,995.00  10,953.50 168,469.60


### Customer-Level Aggregation

In [20]:
customer_summary = (
    customer_df.groupby("CustomerID")
    .agg(
        TotalOrders=("InvoiceNo", "nunique"),
        TotalItems=("Quantity", "sum"),
        TotalRevenue=("Revenue", "sum")
    )
    .reset_index()
)

print(customer_summary.head())

   CustomerID  TotalOrders  TotalItems  TotalRevenue
0   12,346.00           12       74285     77,556.46
1   12,347.00            8        2967      4,921.53
2   12,348.00            5        2714      2,019.40
3   12,349.00            4        1624      4,428.69
4   12,350.00            1         197        334.40


## Final Data Quality Check

In [21]:
print("=" * 60)
print("FINAL DATA QUALITY CHECK")
print("=" * 60)

print("Rows:", len(customer_df))
print("Columns:", customer_df.shape[1])
print("Unique customers:", customer_df["CustomerID"].nunique())
print("Missing values:", customer_df.isnull().sum().sum())
print("Duplicates:", customer_df.duplicated().sum())
print("Negative quantity:", (customer_df["Quantity"] < 0).sum())
print("Negative price:", (customer_df["UnitPrice"] < 0).sum())

FINAL DATA QUALITY CHECK
Rows: 779425
Columns: 10
Unique customers: 5878
Missing values: 0
Duplicates: 0
Negative quantity: 0
Negative price: 0


## Export Cleaned Dataset

In [22]:
customer_df.to_csv(
    "../data/processed/cleaned_retail.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
